# News Category Classification — Main Notebook

Multi-class news classifier (42 categories) using headline + short description.

**Pipeline:** Kaggle dataset → preprocessing → TF-IDF + numeric features → 6 classical models + RoBERTa-embedding SVM → Gradio UI with Groq LLM explanations and similarity retrieval.

Per ADR-009, all pipeline code lives in this notebook. The only Python file maintained outside is `src/preprocessing.py` (because its `clean_text()` is unit-tested in CI).

Section headers map directly to sprint-1 tasks; later sprints fill in the remaining models and the full Gradio UI.

## Setup & Imports

Runs once at the top of the notebook. Idempotent — safe to re-run.

In [ ]:
import logging
import os
import sys
from pathlib import Path

# Make `src/` importable when running from the notebooks/ folder.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Load .env if present (local Windows / Linux). On Colab use Colab Secrets UI instead.
try:
    from dotenv import load_dotenv
    load_dotenv(PROJECT_ROOT / ".env")
except ImportError:
    pass

# Standard project paths.
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
EMBEDDINGS_DIR = DATA_DIR / "embeddings"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"
CONFUSION_DIR = REPORTS_DIR / "confusion"
for d in (RAW_DIR, PROCESSED_DIR, EMBEDDINGS_DIR, MODELS_DIR, REPORTS_DIR, CONFUSION_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Single source of truth for randomness (ADR-section: reproducibility).
RANDOM_STATE = 42

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("main")
log.info(f"Project root: {PROJECT_ROOT}")

## 1. Data Loading — S1-T5

Download the News Category Dataset (~209K rows) from Kaggle. The first run hits Kaggle; subsequent runs read from local cache under `data/raw/`.

In [ ]:
"""S1-T5 — Kaggle download with local cache + JSON-Lines parse.

Returns a DataFrame with columns: link, headline, category,
short_description, authors, date. About 209K rows (full file ~80 MB).
First call hits Kaggle; subsequent calls read from `data/raw/`.
"""

import json

import pandas as pd

DATASET_SLUG = "rmisra/news-category-dataset"
DATASET_FILENAME = "News_Category_Dataset_v3.json"


def load_dataset(force_download: bool = False) -> pd.DataFrame:
    cache_path = RAW_DIR / DATASET_FILENAME
    if cache_path.exists() and not force_download:
        log.info(f"Using cached dataset at {cache_path} ({cache_path.stat().st_size / 1e6:.1f} MB)")
    else:
        # KaggleApi reads credentials from KAGGLE_CONFIG_DIR/kaggle.json.
        os.environ["KAGGLE_CONFIG_DIR"] = str(PROJECT_ROOT)
        from kaggle.api.kaggle_api_extended import KaggleApi

        api = KaggleApi()
        api.authenticate()
        log.info(f"Downloading {DATASET_SLUG} from Kaggle...")
        api.dataset_download_files(DATASET_SLUG, path=str(RAW_DIR), unzip=True)
        log.info(f"Cached to {cache_path}")

    rows = []
    with cache_path.open("r", encoding="utf-8") as fh:
        for line in fh:
            rows.append(json.loads(line))
    df = pd.DataFrame(rows)
    log.info(f"Loaded {len(df):,} articles | columns: {list(df.columns)}")
    return df


raw_df = load_dataset()
raw_df.head()

## 2. Preprocessing — S1-T6

Calls `clean_text()` from `src/preprocessing.py` (kept as a module so it can be unit-tested in CI). Adds three numeric features (word / char / punct counts) and writes the cleaned dataset to `data/processed/cleaned.parquet`.

In [ ]:
"""S1-T6 — Apply clean_text + numeric features over the dataset.

Reads `raw_df` from the cell above, produces the cleaned dataset with
the cleaned text column plus three numeric features
(word_count, char_count, punct_count), and persists to
`data/processed/cleaned.parquet`. Cached on subsequent runs.
"""

from tqdm.auto import tqdm

from src.preprocessing import build_numeric_features, clean_text

CLEANED_PATH = PROCESSED_DIR / "cleaned.parquet"

if CLEANED_PATH.exists():
    log.info(f"Using cached cleaned dataset at {CLEANED_PATH}")
    cleaned_df = pd.read_parquet(CLEANED_PATH)
else:
    log.info(f"Cleaning {len(raw_df):,} articles (~3-5 min on a laptop CPU)...")
    combined = (raw_df["headline"].fillna("") + " " + raw_df["short_description"].fillna("")).tolist()

    feats = [build_numeric_features(t) for t in tqdm(combined, desc="numeric")]
    texts = [clean_text(t) for t in tqdm(combined, desc="clean")]

    cleaned_df = pd.DataFrame(
        {
            "text": texts,
            "category": raw_df["category"].values,
            "word_count": [f["word_count"] for f in feats],
            "char_count": [f["char_count"] for f in feats],
            "punct_count": [f["punct_count"] for f in feats],
        }
    )
    cleaned_df.to_parquet(CLEANED_PATH, compression="snappy")
    log.info(f"Saved cleaned dataset to {CLEANED_PATH}")

log.info(f"Cleaned dataset: {len(cleaned_df):,} rows x {cleaned_df.shape[1]} columns")
cleaned_df.head()

## 3. EDA

Exploratory analysis lives in `notebooks/01_eda.ipynb`. Sprint 1 produces part 1 (shape / missing / class distribution); sprint 2 adds the full chart set.

## 4. Classical Features — S1-T8

TF-IDF (uni+bi-grams, capped at 50K features, sublinear TF) stacked with three numeric features (word / char / punct counts, scaled). Persisted to `models/`.

In [ ]:
# Implemented in S1-T8.

## 5. Train Logistic Regression Baseline — S1-T9

Stratified 80/20 split, `class_weight='balanced'`, `GridSearchCV` over `C ∈ {0.1, 1, 10}`. Persists best estimator to `models/logreg_best.joblib`.

In [ ]:
# Implemented in S1-T9.

## 6. Evaluate Models — S1-T10

Computes (accuracy, precision, recall, F1-macro, F1-weighted, ROC-AUC OvR) for each persisted model and appends a row to `reports/model_comparison.csv`. Saves a confusion-matrix PNG to `reports/confusion/<name>.png`.

In [ ]:
# Implemented in S1-T10.

## 7. RoBERTa Feasibility Spike — S1-T11

Embed 100 sample sentences through `roberta-base`, mean-pool, save `(100, 768)` to `data/embeddings/roberta_spike.npy`. Logs peak memory and wall-time so sprint 2 can size the full extraction. **Run on Colab GPU** — local CPU is too slow for the full version.

In [ ]:
# Implemented in S1-T11.

## 8. Groq LLM Smoke Test — S1-T13

Auths against Groq with `GROQ_API_KEY`, sends a one-line prompt, prints the response. Verifies the API path before the full LLM integration in sprint 2.

In [ ]:
# Implemented in S1-T13.

## 9. Minimal Gradio App — S1-T12

Two text inputs (headline, short description), one Predict button. Displays the LogReg baseline's predicted category + confidence. Launches a public share link from Colab. The full UI (LLM explanation + similar articles) ships in sprint 2.

In [ ]:
# Implemented in S1-T12.